In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from colorPalette import colors

mm_to_pt = 72 / 25.4
mm_to_inch = 1 / 25.4
font_size_mm = 2.5
font_size_pt = font_size_mm * mm_to_pt  # ≈ 8.5 pt
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": font_size_pt,
    "axes.labelsize": font_size_pt,
    "axes.titlesize": font_size_pt,
    "xtick.labelsize": font_size_pt,
    "ytick.labelsize": font_size_pt,
    "legend.fontsize": font_size_pt,
})

In [ ]:
n_bump = '06'
csv_dir = '../data'

# Load data
df_perfect = pd.read_csv(csv_dir+'/PV_'+n_bump+'_PV_perfect.csv')
df_imperfect = pd.read_csv(csv_dir+'/PV_'+n_bump+'_PV_imperfect.csv')

# Preserve original ordering of the volume groups
vol_order = df_imperfect['vol_key'].drop_duplicates().tolist()

# Map vol_key -> label (for legend text, matching the original pltLabel scheme)
vol_to_label = (
    df_imperfect.drop_duplicates('vol_key')
    .set_index('vol_key')['vol_label']
    .to_dict()
)

# Plotting
fig = plt.figure(figsize=(60. * mm_to_inch, 60. * mm_to_inch))
ax = fig.add_axes([0.2, 0.175, 0.6, 0.775])

# Near-perfect shell reference
ax.plot(
    df_perfect['V_norm'], df_perfect['p_norm_mean'],
    'k-', label='near-perfect shell',
    color=colors['grey'][2], zorder=-1
)
ax.fill_between(
    df_perfect['V_norm'],
    df_perfect['p_norm_mean'] - df_perfect['p_norm_std'],
    df_perfect['p_norm_mean'] + df_perfect['p_norm_std'],
    color=colors['grey'][2],
    alpha=0.2,
    linewidth=0,
    zorder=-1
)

# Imperfect shells, one curve per excitation volume
for i, vol in enumerate(vol_order):
    grp = df_imperfect[df_imperfect['vol_key'] == vol]

    pltLabelPre = 'Vol=' if i == 0 else ''
    shadeLabel = 's.d.' if i == len(vol_order) - 1 else ''

    ax.plot(
        grp['V_norm'], grp['p_norm_mean'],
        'k-',
        label=pltLabelPre + str(vol_to_label[vol]),
        color=colors['blue'][i]
    )
    ax.fill_between(
        grp['V_norm'],
        grp['p_norm_mean'] - grp['p_norm_std'],
        grp['p_norm_mean'] + grp['p_norm_std'],
        color=colors['blue'][i],
        alpha=0.2,
        label=shadeLabel,
        linewidth=0
    )

ax.set_xlabel(r"Norm. shell volume, $V_\mathrm{norm}$")
ax.set_ylabel(r"Norm. pressure, $p_\mathrm{norm}$")
ax.set_xlim([0., 0.0075])
ax.set_ylim([0., 1.])
ax.set_xticks([0., 0.0025, 0.005, 0.0075])
ax.set_yticks([0., 0.2, 0.4, 0.6, 0.8, 1.])
ax.grid(True, alpha=0.3)
ax.legend(
    frameon=False,
    handlelength=1.2,
    handletextpad=0.4,
    borderpad=0,
    labelspacing=0.2
)
plt.tight_layout(pad=0.2)
plt.show()

In [ ]:

# Load plot-ready data
pts = pd.read_csv(csv_dir+'/PV_'+n_bump+'_stiffness_points.csv')
fit = pd.read_csv(csv_dir+'/PV_'+n_bump+'_stiffness_fit.csv')

# Row 0 is the perfect-shell baseline; the rest are the imperfect shells
pts_perfect   = pts.iloc[:1]
pts_imperfect = pts.iloc[1:]

pltLabel = pts['vol_label'].tolist()

# Plotting
fig = plt.figure(figsize=(60. * mm_to_inch, 60. * mm_to_inch))
ax1 = fig.add_axes([0.2, 0.175, 0.6, 0.775])
ax2 = ax1.twinx()

# Perfect-shell baseline (grey markers at x=0)
ax1.errorbar(
    pts_perfect['x_index'], pts_perfect['max_pressure'],
    yerr=pts_perfect['max_pressure_sd'],
    fmt='x', markersize=5, elinewidth=1, capsize=3,
    color=colors['grey'][0],
)
ax2.errorbar(
    pts_perfect['x_index'], pts_perfect['stiffness'],
    yerr=pts_perfect['stiffness_sd'],
    fmt='o', markersize=3, elinewidth=1, capsize=3,
    color=colors['grey'][2],
)

# Imperfect shells (blue markers at x=1,2,3)
l1 = ax1.errorbar(
    pts_imperfect['x_index'], pts_imperfect['max_pressure'],
    yerr=pts_imperfect['max_pressure_sd'],
    fmt='x', markersize=5, elinewidth=1, capsize=3,
    color=colors['blue'][0],
    label='Knock-down factor'
)
l2 = ax2.errorbar(
    pts_imperfect['x_index'], pts_imperfect['stiffness'],
    yerr=pts_imperfect['stiffness_sd'],
    fmt='o', markersize=3, elinewidth=1, capsize=3,
    color=colors['blue'][2],
    label='Stiffness'
)

# Exponential-decay fit (dashed grey line)
ax1.plot(
    fit['x_index'], fit['fit_pressure'],
    '--', color='#999999', linewidth=1.
)

ax1.set_ylim([0., 1.])
ax1.set_ylabel(r"Norm. pressure, $p_\mathrm{norm}$")
ax2.set_ylim([0., 150.])
ax2.set_ylabel(r"Norm. stiffness, $E_\mathrm{norm}$")
ax1.set_xlim([-0.25, 3.25])
ax1.set_xlabel('Norm. speaker volume')
ax1.set_xticks(pts['x_index'])
ax1.set_xticklabels(pltLabel)
ax2.set_yticks([0., 30., 60., 90., 120., 150.])

handles = [l1, l2]
labels = [h.get_label() for h in handles]
ax1.legend(
    handles, labels, loc=[0.05, 0.05],
    frameon=False, handlelength=1.2, handletextpad=0.4,
    borderpad=0, labelspacing=0.2
)
ax1.grid(True, alpha=0.3)
plt.tight_layout(pad=0.2)
plt.show()